In [1]:
!pip install transformers datasets evaluate accelerate scikit-learn gradio


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score

In [3]:
# checking is cp or gpu is used
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

Torch: 2.7.1+cu118
CUDA: True
cuda


In [4]:
dataset = load_dataset("fancyzhx/ag_news")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [6]:
dataset["train"][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [7]:
dataset["train"].features["label"].names

['World', 'Sports', 'Business', 'Sci/Tech']

In [8]:
# loading tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

In [9]:
# toinizing the text
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

In [10]:
tokenized_train = dataset["train"].map(
    tokenize,
    batched=True
)

tokenized_test = dataset["test"].map(
    tokenize,
    batched=True
)

In [11]:
# removing text after tokenizaton
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

tokenized_train = tokenized_train.rename_column(
    "label",
    "labels"
)

tokenized_test = tokenized_test.rename_column(
    "label",
    "labels"
)

In [12]:
# chamging format to pytorch format
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

In [13]:
# selecting size for training data and teting data
small_train = tokenized_train.shuffle(seed=42).select(range(5000))
small_test = tokenized_test.shuffle(seed=42).select(range(1000))

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [15]:
print(next(model.parameters()).device)

cuda:0


In [16]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [17]:
training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=2,

    weight_decay=0.01,

    fp16=True,

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    logging_steps=100,

    report_to="none"
)

In [18]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=small_train,

    eval_dataset=small_test,

    compute_metrics=compute_metrics
)

In [19]:
# training the model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.511933,0.443463,0.902000,0.902005
2,0.273762,0.437100,0.909000,0.908820


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=2500, training_loss=0.3810238166809082, metrics={'train_runtime': 1120.0663, 'train_samples_per_second': 8.928, 'train_steps_per_second': 2.232, 'total_flos': 328894725120000.0, 'train_loss': 0.3810238166809082, 'epoch': 2.0})

In [20]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.273762,0.437100,2,0.909000,0.908820


{'eval_loss': 0.4371004104614258,
 'eval_accuracy': 0.909,
 'eval_f1': 0.9088201396575569}

In [21]:
print("Accuracy :", results["eval_accuracy"])

print("F1 Score :", results["eval_f1"])

Accuracy : 0.909
F1 Score : 0.9088201396575569


In [22]:
trainer.save_model("saved_model")

tokenizer.save_pretrained("saved_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_model\\tokenizer_config.json', 'saved_model\\tokenizer.json')

In [23]:
import json

labels = dataset["train"].features["label"].names

with open("saved_model/labels.json","w") as f:

    json.dump(labels,f)

In [24]:
text = "Apple unveils its latest AI-powered iPhone with improved performance."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()

with torch.no_grad():

    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Prediction:", labels[prediction])

Prediction: Sci/Tech


In [25]:
news = [
    "Microsoft announces a new AI chip.",
    "Pakistan defeats India in the cricket final.",
    "Oil prices rise due to global demand.",
    "The United Nations holds a climate summit."
]

for text in news:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    print(f"{text}\nPrediction: {labels[pred]}\n")

Microsoft announces a new AI chip.
Prediction: Sci/Tech

Pakistan defeats India in the cricket final.
Prediction: Sports

Oil prices rise due to global demand.
Prediction: Business

The United Nations holds a climate summit.
Prediction: Sci/Tech



In [27]:
!pip install gradio


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import torch
import json
import gradio as gr

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("saved_model")

model = AutoModelForSequenceClassification.from_pretrained("saved_model")

model.to(device)

model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [30]:
with open("saved_model/labels.json", "r") as f:
    labels = json.load(f)

print(labels)

['World', 'Sports', 'Business', 'Sci/Tech']


In [31]:
def predict_news(news):

    inputs = tokenizer(
        news,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    return labels[prediction]

In [32]:
interface = gr.Interface(

    fn=predict_news,

    inputs=gr.Textbox(
        lines=4,
        placeholder="Enter a news headline here..."
    ),

    outputs=gr.Textbox(label="Predicted Category"),

    title="News Topic Classifier Using BERT",

    description="Enter any news headline and BERT will predict its topic."
)

In [33]:
interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
